# 🎓 HFU KI-Praktikum | Versuch 03: Bilderfassung (Data Collection)
## 🎓 HFU AI Lab | Experiment 03: Image Acquisition

**Ziel / Objective:**
- 🇩🇪 Erfassung von mindestens 30 Bildern der Prüfobjekte (Szenario 1: OP-Besteck ODER Szenario 2: Bosch IXO Akkuschrauber Bauteile) unter konstantem Licht.
- 🇬🇧 Capturing at least 30 images of target objects (Scenario 1: Surgical Tools OR Scenario 2: Bosch IXO components) under controlled lighting.

### 1. Einrichtung der Ziel-Ordnerstruktur (YOLO Standard Format)

In [ ]:
import os
import cv2
import time
import urllib.request
import numpy as np

DATASET_DIR = "/workspace/student_data/dataset"
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "images/train")
VAL_IMG_DIR = os.path.join(DATASET_DIR, "images/val")

os.makedirs(TRAIN_IMG_DIR, exist_ok=True)
os.makedirs(VAL_IMG_DIR, exist_ok=True)

print(f"[DE] Ziel-Ordner vorbereitet: {DATASET_DIR}")
print(f"[EN] Dataset directory prepared: {DATASET_DIR}")

### 2. Bilder aus dem zentralen Kamera-Stream erfassen

In [ ]:
def get_camera_frame():
    """Holt das aktuelle Frame vom zentralen Kamera-Stream-Server."""
    try:
        stream = urllib.request.urlopen("http://backend:8000/api/stream", timeout=3)
        bytes_data = b''
        for _ in range(100):
            bytes_data += stream.read(1024)
            a = bytes_data.find(b'\xff\xd8')
            b = bytes_data.find(b'\xff\xd9')
            if a != -1 and b != -1:
                jpg = bytes_data[a:b+2]
                frame = cv2.imdecode(np.frombuffer(jpg, dtype=np.uint8), cv2.IMREAD_COLOR)
                if frame is not None:
                    return frame
    except Exception:
        pass
    return None

def capture_sample(count=1, split='train'):
    target_dir = TRAIN_IMG_DIR if split == 'train' else VAL_IMG_DIR
    for i in range(count):
        frame = get_camera_frame()
        if frame is not None:
            filename = f"img_{int(time.time())}_{i:02d}.jpg"
            filepath = os.path.join(target_dir, filename)
            cv2.imwrite(filepath, frame)
            print(f"[OK] Saved ({split}): {filepath}")
            time.sleep(0.5)
        else:
            print("[WARN] Frame capture empty.")

# 5 Trainingsbilder zur Demonstration erfassen
capture_sample(count=5, split='train')